## Read Model Weight and Run Inference

#### How to install library in PYNQ Kria KV260

sudo /usr/local/share/pynq-venv/bin/pip3 install torch --index-url https://download.pytorch.org/whl/cpu \
sudo /usr/local/share/pynq-venv/bin/pip3 install scikit-learn

In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
from sklearn.preprocessing import StandardScaler
import copy

In [2]:
###########################################################################
# Re-define the model architecture (must match the saved model)
###########################################################################

class FoodPreferenceANN(nn.Module):
    def __init__(self, input_dim=4, hidden_dim=8):
        super(FoodPreferenceANN, self).__init__()
        self.layer1 = nn.Linear(input_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.layer2 = nn.Linear(hidden_dim, 1)
        
    def forward(self, x):
        out = self.layer1(x)
        out = self.relu(out)
        out = self.layer2(out)
        return out

In [3]:
###########################################################################
# Instantiate and load weights from .pt File
###########################################################################

model = FoodPreferenceANN(input_dim=4, hidden_dim=8)
model.load_state_dict(torch.load('/home/ubuntu/workspace/ai_food_weights.pt'))
model.eval()  # Set model to evaluation mode

print("Model weights successfully loaded from ai_food_weights.pt!")

Model weights successfully loaded from ai_food_weights.pt!


In [4]:
###########################################################################
# Setup scaler (re-fitted on training feature distribution)
###########################################################################

# Note: Features are [Sour, Sweet, Salty, Spicy]
train_features = np.array([
    [1, 6, 5, 1],  # Nasi Tim Ayam
    [2, 4, 6, 7],  # Telur Balado
    [2, 3, 4, 1],  # Sayur Bening Bayam
    [2, 8, 6, 2],  # Ayam Goreng Mentega
    [1, 4, 4, 2],  # Tumis Buncis Wortel
    [1, 5, 5, 2]   # Opor Ayam
])

scaler = StandardScaler()
scaler.fit(train_features);

In [5]:
###########################################################################
# Perform inference on food items using float (original) model
###########################################################################

new_foods = [
    {"Name": "Nasi Tim Ayam",        "Sour": 1, "Sweet": 6, "Salty": 5, "Spicy": 1},
    {"Name": "Telur Balado",         "Sour": 2, "Sweet": 4, "Salty": 6, "Spicy": 7},
    {"Name": "Sayur Bening Bayam",   "Sour": 2, "Sweet": 3, "Salty": 4, "Spicy": 1},
    {"Name": "Ayam Goreng Mentega", "Sour": 2, "Sweet": 8, "Salty": 6, "Spicy": 2},
    {"Name": "Tumis Buncis Wortel",  "Sour": 1, "Sweet": 4, "Salty": 4, "Spicy": 2},
    {"Name": "Opor Ayam",            "Sour": 1, "Sweet": 5, "Salty": 5, "Spicy": 2}
]

# Extract feature matrix [Sour, Sweet, Salty, Spicy]
X_new = np.array([[f['Sour'], f['Sweet'], f['Salty'], f['Spicy']] for f in new_foods])

# Scale inputs and convert to PyTorch tensor
X_new_scaled = scaler.transform(X_new)
X_tensor = torch.tensor(X_new_scaled, dtype=torch.float32)

# Run model inference
with torch.no_grad():
    raw_logits = model(X_tensor)
    probabilities = torch.sigmoid(raw_logits)
    predictions = (probabilities >= 0.5).float()

print("\n--- Inference Results ---")
for food, prob, pred in zip(new_foods, probabilities, predictions):
    taste = "Like" if pred.item() == 1.0 else "Dislike"
    print(f"Food: {food['Name']:22s} | Prob(Like): {prob.item():.4f} | Prediction: {taste}")


--- Inference Results ---
Food: Nasi Tim Ayam          | Prob(Like): 0.9892 | Prediction: Like
Food: Telur Balado           | Prob(Like): 0.0067 | Prediction: Dislike
Food: Sayur Bening Bayam     | Prob(Like): 0.0003 | Prediction: Dislike
Food: Ayam Goreng Mentega    | Prob(Like): 0.9961 | Prediction: Like
Food: Tumis Buncis Wortel    | Prob(Like): 0.0387 | Prediction: Dislike
Food: Opor Ayam              | Prob(Like): 0.9408 | Prediction: Like


## Run Inference with FPGA Class

In [6]:
from pynq import Overlay
from pynq import allocate
import numpy as np

In [7]:
###########################################################################
# FPGA systolic matrix multiplication wrapper functions
###########################################################################

def init_fpga(bitstream_path='/home/ubuntu/workspace/systolic_mmult.bit'):
    global overlay, dma, dma_send, dma_recv
    overlay = Overlay(bitstream_path)
    dma = overlay.axi_dma_0
    dma_send = dma.sendchannel
    dma_recv = dma.recvchannel

def pack_matrix(mat):
    """
    Packs a 4x4 matrix of INT8 values into a 1D array of 4 uint32 words.
    Each row (4 x 8-bit elements) is packed as: [elem3 << 24 | elem2 << 16 | elem1 << 8 | elem0]
    """
    # Cast to uint32 to ensure bitwise operations don't overflow signed representation
    mat_u32 = np.asarray(mat, dtype=np.uint8).astype(np.uint32)
    
    shifts = np.array([0, 8, 16, 24], dtype=np.uint32)
    # Shift columns and bitwise-OR along axis 1 (row-wise)
    return np.bitwise_or.reduce(mat_u32 << shifts, axis=1)

def unpack_matrix(buf):
    """
    Unpacks a 1D buffer of 8 uint64 words into a 4x4 matrix of INT32 accumulators.
    Each 64-bit word contains two 32-bit signed values:
      - Low 32 bits (bits 0..31)
      - High 32 bits (bits 32..63)
    8 words x 2 values = 16 elements (4x4 matrix).
    """
    buf64 = np.asarray(buf, dtype=np.uint64)
    
    # Extract lower 32 bits and upper 32 bits from each 64-bit word
    low_32  = (buf64 & 0xFFFFFFFF).astype(np.uint32).view(np.int32)
    high_32 = ((buf64 >> 32) & 0xFFFFFFFF).astype(np.uint32).view(np.int32)
    
    # Interleave low and high 32-bit words to reconstruct the 16 elements
    c_flat = np.empty(16, dtype=np.int32)
    c_flat[0::2] = low_32
    c_flat[1::2] = high_32
    
    return c_flat.reshape(4, 4)

def systolic_mmult(a, b):
    """
    Executes 4x4 matrix multiplication A @ B on hardware.
    - Inputs: a [4x4], b [4x4] (INT8)
    - Output: c [4x4] (INT32)
    """
    # 1. Allocate contiguous memory for DMA
    inp_buf = allocate(shape=(8,), dtype=np.uint32)  # 4 words B + 4 words A
    out_buf = allocate(shape=(8,), dtype=np.uint64)  # 8 words (16 x 32-bit outputs)
    
    # 2. Pack matrices: B first, then A (matches your working input_buffer)
    packed_b = pack_matrix(b)
    packed_a = pack_matrix(a)
    inp_buf[:4] = packed_b
    inp_buf[4:] = packed_a
    
    # 3. DMA Transfer
    dma_send.transfer(inp_buf)
    dma_recv.transfer(out_buf)
    dma_send.wait()
    dma_recv.wait()
    
    # 4. Unpack 64-bit output buffer into 4x4 INT32 matrix
    c = unpack_matrix(out_buf)
    
    # 5. Free DMA memory buffers
    del inp_buf, out_buf
    
    return c

def block_matmul(a, b, block_size=4):
    # Get the dimensions of the input matrices
    m, n_a = a.shape
    n_b, p = b.shape
    if n_a != n_b:
        raise ValueError("Matrix dimensions must be compatible for multiplication")

    # Padding matrix a
    num_row_pad_a = int(np.ceil(m / block_size) * block_size - m)
    num_col_pad_a = int(np.ceil(n_a / block_size) * block_size - n_a)
    padded_row_a = np.vstack([a, np.zeros((num_row_pad_a, a.shape[1]))])
    padded_a = np.hstack([padded_row_a, np.zeros((padded_row_a.shape[0], num_col_pad_a))])

    # Padding matrix b
    num_row_pad_b = int(np.ceil(n_b / block_size) * block_size - n_b)
    num_col_pad_b = int(np.ceil(p / block_size) * block_size - p)
    padded_row_b = np.vstack([b, np.zeros((num_row_pad_b, b.shape[1]))])
    padded_b = np.hstack([padded_row_b, np.zeros((padded_row_b.shape[0], num_col_pad_b))])
    
    # Result matrix c with padding
    c = np.zeros((padded_a.shape[0], padded_b.shape[1]))

    # Iterate over blocks (using padded matrices!)
    for i in range(0, padded_a.shape[0], block_size):
        for j in range(0, padded_b.shape[1], block_size):
            for k in range(0, padded_a.shape[1], block_size):
                A_block = padded_a[i:i+block_size, k:k+block_size]
                B_block = padded_b[k:k+block_size, j:j+block_size]

                # Multiply the blocks
                C_block = systolic_mmult(A_block.astype(np.int8), B_block.astype(np.int8))

                # Accumulate the result
                c[i:i+block_size, j:j+block_size] += C_block

    # Remove padding
    return c[:m, :p]

In [8]:
###########################################################################
# Program FPGA
###########################################################################

init_fpga()

In [9]:
###########################################################################
# FPGA linear class
###########################################################################

class FPGAQuantizedLinear(nn.Module):

    def __init__(self, original_layer, act_scale):
        super().__init__()
        self.in_features = original_layer.in_features
        self.out_features = original_layer.out_features

        # Layer-specific activation scale
        self.act_scale = act_scale

        # Layer-specific weight scale
        weights_float = original_layer.weight.data.detach().cpu().numpy()
        self.weight_scale = np.max(np.abs(weights_float)) / 127
        self.weights_int8 = np.round(weights_float / self.weight_scale).astype(
            np.int8
        )

    def forward(self, x):
        # --- PRE-PROCESSING (Simulating CPU -> FPGA Transfer) ---
        # Scale and convert input to int8
        x_np = x.detach().cpu().numpy()
        x_int8 = np.clip(x_np / self.act_scale * 127, -128, 127).astype(np.int8)

        # --- DEBUG PRINT ---
        # x1 is the input activations, x2 is the transposed weights
        x1_shape = x_int8.shape
        x2_shape = self.weights_int8.T.shape
        print(f"DEBUG: matmul x1 {x1_shape} @ x2 {x2_shape}")
        # print(f"    x1 = {x_int8}")
        # print(f"    x2 = {self.weights_int8.T}")
        # print(f"==============================================")

        # --- THE Systolic Hardware Core ---
        out_int32 = block_matmul(x_int8.astype(np.int32), self.weights_int8.T.astype(np.int32))

        # --- POST-PROCESSING (Simulating FPGA -> CPU Transfer) ---
        # Convert back to float using both scales
        total_scale = (self.act_scale / 127) * self.weight_scale
        out_float = out_int32.astype(np.float32) * total_scale

        return torch.from_numpy(out_float).to(x.device)

In [10]:
###########################################################################
# Utility functions to modify the model
###########################################################################

def integrate_numpy_quantization(model, act_scale):
    """
    Recursively replaces all nn.Linear layers with NumpyQuantizedLinear.
    """
    for name, module in model.named_children():
        if isinstance(module, nn.Linear):
            # 1. Create the new quantized layer using the original layer's data
            quantized_layer = FPGAQuantizedLinear(module, act_scale)

            # 2. Replace the old layer with the new one in the model
            setattr(model, name, quantized_layer)
            print(f"Replaced layer: {name}")
        else:
            # 3. If it's a container (like Sequential or MultiheadAttention),
            # recurse into its children
            integrate_numpy_quantization(module, act_scale)

def get_calibration_scale(model, calibration_loader):
    model.eval()
    max_values = []

    with torch.no_grad():
        for batch in calibration_loader:
            # 1. Unpack the Tensor from the DataLoader list/tuple
            x = batch[0] if isinstance(batch, (list, tuple)) else batch

            # 2. Get model outputs (single Tensor output)
            logits = model(x)

            # 3. Track maximum activation magnitude across the batch
            max_values.append(torch.max(torch.abs(logits)).item())

    # Return the average maximum activation
    return sum(max_values) / len(max_values)

In [11]:
###########################################################################
# Get activation scale
###########################################################################

# Transform the raw training features using the fitted scaler
X_scaled = scaler.transform(train_features)
# Convert the scaled NumPy array into a PyTorch FloatTensor
X_tensor = torch.tensor(X_scaled, dtype=torch.float32)
# Create TensorDataset and wrap it in a DataLoader
# (batch_size can be set to your preferred batch size, e.g., 2 or 6)
calibration_dataset = TensorDataset(X_tensor)
calibration_loader = DataLoader(
    calibration_dataset, batch_size=2, shuffle=False
)

# Find the activation scale from floating-point calibration runs
act_scale = get_calibration_scale(model, calibration_loader)
print(f"Calculated Activation Scale: {act_scale}")

Calculated Activation Scale: 5.428282419840495


In [12]:
###########################################################################
# Replace the nn.Linear class
###########################################################################

# Create an isolated deep copy for quantization
model_quantized = copy.deepcopy(model)

# Verify layer before modification
print("Before (Original Model Layer1): ", model.layer1)
print("Before (Quantized Copy Layer1): ", model_quantized.layer1)

# Apply quantization ONLY to the copied model
integrate_numpy_quantization(model_quantized, act_scale)

print("\n--- After Quantization ---")
# Verify layer after modification
print("Original model remains float:   ", model.layer1)
print("Quantized model replaced layer: ", model_quantized.layer1)

Before (Original Model Layer1):  Linear(in_features=4, out_features=8, bias=True)
Before (Quantized Copy Layer1):  Linear(in_features=4, out_features=8, bias=True)
Replaced layer: layer1
Replaced layer: layer2

--- After Quantization ---
Original model remains float:    Linear(in_features=4, out_features=8, bias=True)
Quantized model replaced layer:  FPGAQuantizedLinear()


In [13]:
###########################################################################
# Perform inference on food items using quantized FPGA model
###########################################################################

new_foods = [
    {"Name": "Nasi Tim Ayam", "Sour": 1, "Sweet": 6, "Salty": 5, "Spicy": 1},
    {"Name": "Telur Balado", "Sour": 2, "Sweet": 4, "Salty": 6, "Spicy": 7},
    {"Name": "Sayur Bening Bayam", "Sour": 2, "Sweet": 3, "Salty": 4, "Spicy": 1},
    {"Name": "Ayam Goreng Mentega", "Sour": 2, "Sweet": 8, "Salty": 6, "Spicy": 2},
    {"Name": "Tumis Buncis Wortel", "Sour": 1, "Sweet": 4, "Salty": 4, "Spicy": 2},
    {"Name": "Opor Ayam", "Sour": 1, "Sweet": 5, "Salty": 5, "Spicy": 2},
]

# Extract and scale inputs
X_new = np.array(
    [[f["Sour"], f["Sweet"], f["Salty"], f["Spicy"]] for f in new_foods]
)
X_new_scaled = scaler.transform(X_new)
X_tensor = torch.tensor(X_new_scaled, dtype=torch.float32)

# Set quantized model to evaluation mode
model_quantized.eval()

with torch.no_grad():
    # Run quantized model inference
    raw_logits_quant = model_quantized(X_tensor)

    # Convert NumPy output to PyTorch Tensor if your custom layer returns NumPy arrays
    if isinstance(raw_logits_quant, np.ndarray):
        raw_logits_quant = torch.from_numpy(raw_logits_quant).float()

    probabilities_quant = torch.sigmoid(raw_logits_quant)
    predictions_quant = (probabilities_quant >= 0.5).float()

# Print Quantized Model Results
print("\n--- Quantized Model Inference Results ---")
for food, prob, pred in zip(new_foods, probabilities_quant, predictions_quant):
    taste = "Like" if pred.item() == 1.0 else "Dislike"
    print(
        f"Food: {food['Name']:22s} | Quant Prob(Like): {prob.item():.4f} | Prediction: {taste}"
    )

DEBUG: matmul x1 (6, 4) @ x2 (4, 8)
DEBUG: matmul x1 (6, 8) @ x2 (8, 1)

--- Quantized Model Inference Results ---
Food: Nasi Tim Ayam          | Quant Prob(Like): 0.9144 | Prediction: Like
Food: Telur Balado           | Quant Prob(Like): 0.0150 | Prediction: Dislike
Food: Sayur Bening Bayam     | Quant Prob(Like): 0.0010 | Prediction: Dislike
Food: Ayam Goreng Mentega    | Quant Prob(Like): 0.9704 | Prediction: Like
Food: Tumis Buncis Wortel    | Quant Prob(Like): 0.0284 | Prediction: Dislike
Food: Opor Ayam              | Quant Prob(Like): 0.7157 | Prediction: Like


In [14]:
###########################################################################
# Quantization error calculation
###########################################################################

model_quantized.eval()

with torch.no_grad():
    # Run quantized model inference
    raw_logits_quant = model_quantized(X_tensor)

    # Ensure output is a PyTorch Tensor
    if isinstance(raw_logits_quant, np.ndarray):
        raw_logits_quant = torch.from_numpy(raw_logits_quant).float()

    probabilities_quant = torch.sigmoid(raw_logits_quant)
    predictions_quant = (probabilities_quant >= 0.5).float()

print("\n--- Quantized Model Inference & Probability Comparison ---")
print(
    f"{'Food Name':22s} | {'Float Prob':>10s} | {'Quant Prob':>10s} | {'Abs Error':>10s} | {'Prediction':>10s}"
)
print("-" * 72)

errors = []
for food, prob_orig, prob_q, pred in zip(
    new_foods, probabilities, probabilities_quant, predictions_quant
):
    orig_val = prob_orig.item()
    quant_val = prob_q.item()
    abs_err = abs(orig_val - quant_val)
    errors.append(abs_err)

    taste = "Like" if pred.item() == 1.0 else "Dislike"
    print(
        f"{food['Name']:22s} | {orig_val:10.4f} | {quant_val:10.4f} | {abs_err:10.4f} | {taste:>10s}"
    )

mae = np.mean(errors)
max_err = np.max(errors)
print("-" * 72)
print(f"Mean Absolute Error (MAE):     {mae:.6f}")
print(f"Max Absolute Error:            {max_err:.6f}")

DEBUG: matmul x1 (6, 4) @ x2 (4, 8)
DEBUG: matmul x1 (6, 8) @ x2 (8, 1)

--- Quantized Model Inference & Probability Comparison ---
Food Name              | Float Prob | Quant Prob |  Abs Error | Prediction
------------------------------------------------------------------------
Nasi Tim Ayam          |     0.9892 |     0.9144 |     0.0748 |       Like
Telur Balado           |     0.0067 |     0.0150 |     0.0083 |    Dislike
Sayur Bening Bayam     |     0.0003 |     0.0010 |     0.0007 |    Dislike
Ayam Goreng Mentega    |     0.9961 |     0.9704 |     0.0257 |       Like
Tumis Buncis Wortel    |     0.0387 |     0.0284 |     0.0103 |    Dislike
Opor Ayam              |     0.9408 |     0.7157 |     0.2251 |       Like
------------------------------------------------------------------------
Mean Absolute Error (MAE):     0.057493
Max Absolute Error:            0.225126
